In [1]:
# --- Core Python ---
import os
import warnings
from datetime import datetime
import datetime as dt
import collections

# --- Scientific Computing ---
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xcdat as xc
import xskillscore as xs
import metpy.calc as mpcalc
from scipy.stats import pearsonr
from scipy.signal import butter, filtfilt, sosfilt, lfilter
from skimage.feature import peak_local_max

# --- Dask ---
import dask
import dask.array as da
from dask import delayed, compute
from dask.distributed import Client, LocalCluster

# --- Plotting ---
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams
from matplotlib.patches import Polygon
from matplotlib import ticker
import matplotlib.colors as mcolors
import matplotlib.path as mpath
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec

# --- Cartopy ---
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker

# --- Visualization Utilities ---
import geocat.viz.util as gvutil
import geocat.viz as gv
import cmaps as gvcmaps  # optional custom colormaps

# --- Tropical Cyclone Tools ---
from tropycal import tracks, utils

# --- Warnings ---
warnings.filterwarnings("ignore")

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [ ]:
def read_dart_obs_seq(var, var_dict, date, path, file):
    """
    Read a DART obs_seq NetCDF file with aggressive Dask optimization.
    Keeps all data lazy until final compute; avoids loading full location array.
    """
    rpath = os.path.join(path, file)
    dr = xc.open_mfdataset(rpath, decode_times=False, chunks={'obs_num': 'auto'})

    # --- Metadata strings (small, safe to load) ---
    QCMetaData       = np.array([c.decode('utf-8').strip() for c in dr['QCMetaData'].values])
    CopyMetaData     = np.array([c.decode('utf-8').strip() for c in dr['CopyMetaData'].values])
    ObsTypesMetaData = np.array([c.decode('utf-8').strip() for c in dr['ObsTypesMetaData'].values])

    ind_qc   = np.where(QCMetaData   == var_dict['QCString'])[0]
    ind_copy = np.where(CopyMetaData == var_dict['CopyString'])[0]

    reg_xyz = np.array(var_dict['region'].split(' '), dtype=float)

    # --- Create spatial mask on Dask arrays WITHOUT loading location into RAM ---
    # Extract individual coordinates as Dask arrays (chunked, lazy)
    lon_da = dr['location'][:, 0]
    lat_da = dr['location'][:, 1]
    lev_da = dr['location'][:, 2]

    # Build boolean mask on lazy Dask arrays
    mask_da = (
        (lon_da >= reg_xyz[0]) & (lon_da <= reg_xyz[1]) &
        (lat_da >= reg_xyz[2]) & (lat_da <= reg_xyz[3])
    )

    # Get indices where mask is True (still lazy)
    ind_loc = da.where(mask_da, size=mask_da.sum())[0]  # Dask array of indices

    # --- Compute all required data in ONE fused graph evaluation ---
    (time, which_vert, obs_type, obs_keys,
     lon_subset, lat_subset, lev_subset,
     observations_full, qc_full,
     ObsTypes, ObsIndex, ind_loc_computed) = compute(
        dr['time'].to_dask_array(dim='obs_num')[ind_loc],
        dr['which_vert'].to_dask_array(dim='obs_num')[ind_loc],
        dr['obs_type'].to_dask_array(dim='obs_num')[ind_loc],
        dr['obs_keys'].to_dask_array(dim='obs_num')[ind_loc],
        lon_da[ind_loc],
        lat_da[ind_loc],
        lev_da[ind_loc],
        dr['observations'].to_dask_array(dims=('copy', 'obs_num'))[:, ind_loc],
        dr['qc'].to_dask_array(dims=('QC', 'obs_num'))[:, ind_loc],
        dr['ObsTypes'].to_dask_array(dim='obs_type_dim'),
        dr['ObsIndex'].to_dask_array(dim='obs_num')[ind_loc],
        ind_loc,
    )

    # Convert to numpy if needed
    time        = np.asarray(time)
    which_vert  = np.asarray(which_vert)
    obs_type    = np.asarray(obs_type)
    obs_keys    = np.asarray(obs_keys)
    lon         = np.asarray(lon_subset)
    lat         = np.asarray(lat_subset)
    lev         = np.asarray(lev_subset)
    observations_full = np.asarray(observations_full)
    qc_full     = np.asarray(qc_full)
    ObsTypes    = np.asarray(ObsTypes)
    ObsIndex    = np.asarray(ObsIndex)

    # --- Minimal info output (no expensive min/max on full arrays) ---
    print(f"Loaded {len(time)} observations in region {reg_xyz}")
    print(f"obs_type shape: {obs_type.shape}, observations shape: {observations_full.shape}")

    # Extract copy/QC columns
    observations = observations_full[ind_copy, :]
    qc           = qc_full[ind_qc, :]

    if observations.ndim > 1:
        observations = observations[0, :]
    if qc.ndim > 1:
        qc = qc[0, :]

    return (time, lat, lon, lev, which_vert, ObsTypesMetaData,
            ObsTypes, ObsIndex, obs_type, obs_keys, qc, observations)

In [3]:
def extract_exp_info():
    exp_dict  = {
                 'DARTEN10': {
                             'run': 'DARTEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy',
                             'key': 'dart_en10',
                             'diag1': 'obs_seq',
                             'diag2': 'obs_diag',
                             'diag3': 'closest_member'
                 },
                 'DARTEN20': {
                             'run': 'DARTEN20_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy',
                             'key': 'dart_en20',
                             'diag1': 'obs_seq',
                             'diag2': 'obs_diag',
                             'diag3': 'closest_member'
                 },
                 'DARTEN40': {
                             'run': 'DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy',
                             'key': 'dart_en40',
                             'diag1': 'obs_seq',
                             'diag2': 'obs_diag',
                             'diag3': 'closest_member'
                 },
                 'DARTEN80': {
                             'run': 'DARTEN80_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy',
                             'key': 'dart_en80',
                             'diag1': 'obs_seq',
                             'diag2': 'obs_diag',
                             'diag3': 'closest_member'
                 },
    }
    
    return exp_dict

In [4]:
def define_region(regnam='global'):
    reg_dict = {'global':[(-90,90),(-180,180)],
                'Atlantic':[(10,90),(-100,10)],
                'CONUS':[(25,50),(235,295)],
                'Antarctic':[(-90,-50),(-180,180)],
                'PolarN':[(50,90),(-180,180)],
                #'Greenland':[(59,83),(-74,-11)],
                'Greenland':[(60,85),(-75,-10)],
               }
    return reg_dict[regnam]

In [5]:
def extract_obs_group():
    obs_group = {'Conventional' : 
                     ['TEMPERATURE','SPECIFIC_HUMIDITY','PRESSURE', 
                      'RADIOSONDE_U_WIND_COMPONENT','RADIOSONDE_V_WIND_COMPONENT',
                      'RADIOSONDE_GEOPOTENTIAL_HGT','RADIOSONDE_SURFACE_PRESSURE','RADIOSONDE_TEMPERATURE',
                      'RADIOSONDE_SPECIFIC_HUMIDITY', 'DROPSONDE_U_WIND_COMPONENT','DROPSONDE_V_WIND_COMPONENT',
                      'DROPSONDE_SURFACE_PRESSURE', 'DROPSONDE_TEMPERATURE', 'DROPSONDE_SPECIFIC_HUMIDITY',
                      'AIRCRAFT_U_WIND_COMPONENT', 'AIRCRAFT_V_WIND_COMPONENT','AIRCRAFT_TEMPERATURE',
                      'AIRCRAFT_SPECIFIC_HUMIDITY','ACARS_U_WIND_COMPONENT','ACARS_V_WIND_COMPONENT',
                      'ACARS_TEMPERATURE','ACARS_SPECIFIC_HUMIDITY','RADIOSONDE_SURFACE_ALTIMETER',
                      'DROPSONDE_SURFACE_ALTIMETER','METAR_ALTIMETER','MESONET_SURFACE_ALTIMETER',
                      'MARINE_SFC_U_WIND_COMPONENT','MARINE_SFC_V_WIND_COMPONENT','MARINE_SFC_TEMPERATURE',
                      'MARINE_SFC_SPECIFIC_HUMIDITY','MARINE_SFC_PRESSURE','LAND_SFC_U_WIND_COMPONENT',
                      'LAND_SFC_V_WIND_COMPONENT','LAND_SFC_TEMPERATURE','LAND_SFC_SPECIFIC_HUMIDITY',
                      'LAND_SFC_PRESSURE','MARINE_SFC_ALTIMETER','LAND_SFC_ALTIMETER' ],
                 'Satellite' :
                     ['GPSRO_REFRACTIVITY','SAT_TEMPERATURE','SAT_TEMPERATURE_ELECTRON','SAT_TEMPERATURE_ION',
                      'SAT_DENSITY_NEUTRAL_O3P', 'SAT_DENSITY_NEUTRAL_O2', 'SAT_DENSITY_NEUTRAL_N2',
                      'SAT_DENSITY_NEUTRAL_N4S', 'SAT_DENSITY_NEUTRAL_NO', 'SAT_DENSITY_NEUTRAL_N2D',
                      'SAT_DENSITY_NEUTRAL_N2P', 'SAT_DENSITY_NEUTRAL_H',  'SAT_DENSITY_NEUTRAL_HE',
                      'SAT_DENSITY_NEUTRAL_CO2', 'SAT_DENSITY_NEUTRAL_O1D', 'SAT_DENSITY_ION_O4SP', 
                      'SAT_DENSITY_ION_O2P', 'SAT_DENSITY_ION_N2P', 'SAT_DENSITY_ION_NP', 
                      'SAT_DENSITY_ION_O2DP', 'SAT_DENSITY_ION_O2PP', 'SAT_DENSITY_ION_HP',
                      'SAT_DENSITY_ION_HEP', 'SAT_DENSITY_ION_E', 'SAT_VELOCITY_U',
                      'SAT_DENSITY_ION_NOP','SAT_VELOCITY_V', 'SAT_VELOCITY_W',
                      'SAT_VELOCITY_U_ION','SAT_VELOCITY_V_ION', 'SAT_VELOCITY_W_ION',
                      'SAT_VELOCITY_VERTICAL_O3P','SAT_VELOCITY_VERTICAL_O2', 'SAT_VELOCITY_VERTICAL_N2',
                      'SAT_VELOCITY_VERTICAL_N4S', 'SAT_VELOCITY_VERTICAL_NO', 'SAT_F107','SAT_RHO', 'GPS_PROFILE', 
                      'COSMIC_ELECTRON_DENSITY', 'GND_GPS_VTEC','CHAMP_DENSITY','MIDAS_TEC','SSUSI_O_N2_RATIO',
                      'GPS_VTEC_EXTRAP', 'SABER_TEMPERATURE', 'AURAMLS_TEMPERATURE', 
                      'SAT_U_WIND_COMPONENT', 'SAT_V_WIND_COMPONENT', 'ATOV_TEMPERATURE','AIRS_TEMPERATURE',
                      'AIRS_SPECIFIC_HUMIDITY','GPS_PRECIPITABLE_WATER', 'VADWND_U_WIND_COMPONENT',
                      'VADWND_V_WIND_COMPONENT','CIMMS_AMV_U_WIND_COMPONENT','CIMMS_AMV_V_WIND_COMPONENT'],
                }
    
    return obs_group 

In [6]:
def draw_obs_location_2d(date,var,var_dict,data_dict,fig_path,fgw=35,fgh=25,hs=0.5,ws=0.5):

    #set the extent of the map
    extent = [float(var) for var in var_dict['region'].split(' ')[0:4]]
    if extent[0] >= 0 and extent[1] >= 180:
        extent[0] = extent[0] - 180.0
        extent[1] = extent[1] - 180.0
        
    nrows = 1
    ncols = len(data_dict.keys())
    
    data_projection = ccrs.PlateCarree() #ccrs.Robinson() 
    kwtrans = dict(central_longitude=0.)
    trans = ccrs.PlateCarree(**kwtrans) #ccrs.Stereographic(**kwtrans)
    
    #create figure  
    frac = 12.0/35.0
    frac = fgh/fgw
    fontz = 14 * fgw*1.0/fgh * frac
    fig, axes = plt.subplots(nrows=nrows,
                             ncols=ncols, 
                             figsize=(fgw,fgh), 
                             subplot_kw={'projection': data_projection})
    
    cmap = {
        'blue':    '#377eb8', 
        'orange':  '#ff7f00',
        'green':   '#4daf4a',
        'pink':    '#f781bf',
        'brown':   '#a65628',
        'purple':  '#984ea3',
        'gray':    '#999999',
        'red':     '#e41a1c',
        'yellow':  '#dede00'
    } 
 

    #for i,model in enumerate(data_dict.keys()):
    #    print(model)
    #print(xxx)
        
    for i,model in enumerate(data_dict.keys()):
        bins = data_dict[model].keys()
        colors = list(cmap.keys())[0:len(bins)]
        # increasing sizes for the markers in each bin
        sizes = np.linspace(15, 25, len(bins)) / 3.5
        axes.flat[i].set_extent(extent, ccrs.PlateCarree())

        # Draw land
        axes.flat[i].add_feature(cfeature.LAND, color='silver', zorder=0)
        axes.flat[i].add_feature(cfeature.LAKES, color='white', zorder=0)

        # Use geocat.viz.util convenience function to set axes tick values
        gv.set_axes_limits_and_ticks(axes.flat[i],
                                     xticks=np.arange(extent[0], extent[1]+1, 60),
                                     yticks=np.arange(extent[2], extent[3]+1, 30))

        # Use geocat.viz.util convenience function to make latitude and longitude tick
        # labels
        gv.add_lat_lon_ticklabels(axes.flat[i])
    
        # Removing degree symbol from tick labels to more closely resemble NCL example
        #ax.yaxis.set_major_formatter(LatitudeFormatter(degree_symbol=''))
        #ax.xaxis.set_major_formatter(LongitudeFormatter(degree_symbol=''))

        # Use geocat.viz.util convenience function to add minor and major tick lines
        gv.add_major_minor_ticks(axes.flat[i],
                                 x_minor_per_major=1,
                                 y_minor_per_major=1,
                                 labelsize=fontz)

        # Use geocat.viz.util convenience function to add titles
        #gv.set_titles_and_labels(
        #    ax,
        #    maintitlefontsize=fontz*1.1,
        #    maintitle=
        #    "Dummy station data colored and\nsized according to range of values")

        # Plot markers with values less than first bin value
        masked_dict = dict() 
        labels = []
        for j,x in enumerate(bins):
            y = x.upper()
            if y not in labels:
                labels.append("{}".format(y))
                for kk,(lon,lat) in enumerate(zip(data_dict[model][x]['lon'],data_dict[model][x]['lat'])):
                    masked_lon = np.where(lon > 180, lon-360, lon)
                    masked_lat = lat
                    if kk == 0: 
                        merged_lat = np.array(masked_lat)
                        merged_lon = np.array(masked_lon)
                    else:
                        label = '' 
                        merged_lat = np.concatenate([merged_lat, masked_lat])
                        merged_lon = np.concatenate([merged_lon, masked_lon])
                if y not in masked_dict:
                    masked_dict[y] = dict() 
                if 'lat' not in masked_dict[y].keys():
                    masked_dict[y]['lat'] = [] 
                if 'lon' not in masked_dict[y].keys():
                    masked_dict[y]['lon'] = [] 
                masked_dict[y]['lat'] = merged_lat
                masked_dict[y]['lon'] = merged_lon
                del(merged_lat,merged_lon,masked_lon,masked_lat)
            else:
                for lon,lat in zip(data_dict[model][x]['lon'],data_dict[model][x]['lat']):
                    masked_lon = np.where(lon > 180, lon-360, lon)
                    masked_lat = lat
                    masked_dict[y]['lat'] = np.concatenate(masked_dict[y]['lat'],masked_lat)
                    masked_dict[y]['lon'] = np.concatenate(masked_dict[y]['lon'],masked_lon)
                    
        ntot = 0 
        for xx in labels:
            ntot = ntot + len( masked_dict[xx]['lon'])
            
        for kk,x in enumerate(labels):
            ratio = len(masked_dict[x]['lon'])*100.0 / ntot
            axes.flat[i].scatter(
                masked_dict[x]['lon'],
                masked_dict[x]['lat'],
                label='{} ({:0.1f}%)'.format(labels[kk],ratio),
                s=sizes[kk],
                color=cmap[colors[kk]],
                alpha=0.5, 
                edgecolors='none',
                zorder=1)
            
        # produce a legend with a cross-section of sizes from the scatter
        #handles, labels = scatter.legend_elements(prop=model, alpha=0.6)
        #legend2 = ax.legend(handles, labels, loc="upper right", title="Sizes")
        #legend1 = axes.flat[i].legend(*scatter.legend_elements(num=len(labels)),
        #                              loc="upper left", title="Ranking")
        #axes.flat[i].add_artist(legend1)

        # `ncol` being equal to half of the number of labels makes the legend appear
        # horizontal with two rows
        nnlab = 3
        if len(labels) <= nnlab:
            bbox_to_anchor = (0.01, -0.25)
            columnspacing = 1.0
        else:
            bbox_to_anchor = (-0.05, -0.35)
            columnspacing = 1.0
        
        axes.flat[i].legend(
            bbox_to_anchor = bbox_to_anchor,
            ncol=nnlab,
            fontsize=fontz*1.05,
            loc='lower left',
            markerscale = 3., 
            scatterpoints = 1,
            columnspacing=columnspacing,
            frameon=False)
        
        # Customize ticks and labels
        axes.flat[i].tick_params(labelsize=fontz*1.1, length=8)
        axes.flat[i].set_xlabel("")
        axes.flat[i].set_ylabel("")      
        axes.flat[i].tick_params(top=False, right=False)

        #axes.flat[i].set_title('{}'.format(var),loc='left',fontsize=fontz*1.2,pad=20)
        #axes.flat[i].set_title('{}'.format(metrics_str),loc='right',fontsize=fontz*1.2,pad=20)
        axes.flat[i].set_title('{} Observation'.format(model.capitalize()),loc='center',fontsize=fontz*1.2,pad=10)
        #axes.flat[i].text(1,1,metrics_str,fontsize=fontz*1.2, color='red', fontweight='bold', ha='center', va='center') #transform=plt.gcf().transFigure)
        #axes.flat[i].text(xpos_str,ypos_str,metrics_str,color='red',fontsize=fontz,fontweight='bold',ha='center',va='center',transform=axes.flat[k].transAxes)
            
    #plt.tight_layout()
    plt.subplots_adjust(hspace=hs)
    plt.subplots_adjust(wspace=ws)
    plt.show()
    
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    fname = os.path.join(fig_path,'fig_2d_obs_{}_{}.png'.format(var,date))
    fig.savefig(fname)
    return 

In [7]:
def draw_obs_location_2d(date, var, var_dict, data_dict, fig_path,
                         fgw=35, fgh=25, hs=0.5, ws=0.5):
    """
    Draw 2D scatter maps of observation locations, grouped by bins.

    Parameters
    ----------
    date : str
        Date string for file naming.
    var : str
        Variable name (e.g., 'PRECT').
    var_dict : dict
        Dictionary containing region info with 'region' key (e.g., '-90 90 -180 180').
    data_dict : dict
        Dictionary with structure: {model: {bin_name: {'lon': [...], 'lat': [...]}}}
    fig_path : str
        Output directory to save the figure.
    fgw, fgh : float
        Width and height of the figure.
    hs, ws : float
        Vertical and horizontal space between subplots.
    """
    #set the extent of the map
    extent = [float(var) for var in var_dict['region'].split(' ')[0:4]]
    if extent[0] >= 0 and extent[1] >= 180:
        extent[0] = extent[0] - 180.0
        extent[1] = extent[1] - 180.0

    ncols = len(data_dict)
    fig, axes = plt.subplots(1, ncols, figsize=(fgw, fgh),
                             subplot_kw={'projection': ccrs.PlateCarree()})
    if ncols == 1:
        axes = [axes]

    frac = fgh / fgw
    fontz = 14 * (fgw / fgh) * frac

    cmap = {
        'blue': '#377eb8', 'orange': '#ff7f00', 'green': '#4daf4a',
        'pink': '#f781bf', 'brown': '#a65628', 'purple': '#984ea3',
        'gray': '#999999', 'red': '#e41a1c', 'yellow': '#dede00'
    }

    for i, model in enumerate(data_dict):
        ax = axes[i]
        ax.set_extent(extent)
        ax.add_feature(cfeature.LAND, color='silver', zorder=0)
        ax.add_feature(cfeature.LAKES, color='white', zorder=0)

        gv.set_axes_limits_and_ticks(ax,
                                     xticks=np.arange(extent[0], extent[1] + 1, 60),
                                     yticks=np.arange(extent[2], extent[3] + 1, 30))
        gv.add_lat_lon_ticklabels(ax)
        gv.add_major_minor_ticks(ax, x_minor_per_major=1, y_minor_per_major=1, labelsize=fontz)

        bins = list(data_dict[model].keys())
        colors = list(cmap.keys())[:len(bins)]
        sizes = np.linspace(15, 25, len(bins)) / 3.5

        total_points = 0
        bin_coords = {}

        for j, bin_key in enumerate(bins):
            lons = [np.where(lon > 180, lon - 360, lon) for lon in data_dict[model][bin_key]['lon']]
            lats = data_dict[model][bin_key]['lat']
            lons = np.concatenate(lons) if isinstance(lons[0], np.ndarray) else np.array(lons)
            lats = np.concatenate(lats) if isinstance(lats[0], np.ndarray) else np.array(lats)
            bin_label = bin_key.upper()
            bin_coords[bin_label] = {'lon': lons, 'lat': lats}
            total_points += len(lons)

        for j, label in enumerate(bin_coords):
            ratio = 100.0 * len(bin_coords[label]['lon']) / total_points
            ax.scatter(bin_coords[label]['lon'], bin_coords[label]['lat'],
                       label=f"{label} ({ratio:.1f}%)",
                       s=sizes[j], color=cmap[colors[j]],
                       alpha=0.5, edgecolors='none', zorder=1)

        nnlab = 3
        bbox_anchor = (0.01, -0.25) if len(bin_coords) <= nnlab else (-0.05, -0.35)
        ax.legend(bbox_to_anchor=bbox_anchor,
                  ncol=nnlab,
                  fontsize=fontz * 1.05,
                  loc='lower left',
                  markerscale=3.0,
                  scatterpoints=1,
                  columnspacing=1.0,
                  frameon=False)

        ax.set_title(f"{model.capitalize()} Observation", loc='center', fontsize=fontz * 1.2, pad=10)
        ax.tick_params(labelsize=fontz * 1.1, length=8)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(top=False, right=False)

    plt.subplots_adjust(hspace=hs, wspace=ws)

    if not os.path.exists(fig_path):
        os.makedirs(fig_path)

    pdf_filename = os.path.join(fig_path, f"fig_2d_obs_{var}_{date}.pdf")
    fig.savefig(pdf_filename)
    plt.close()


In [ ]:
def _collect_group(grp, obs_group, ObsTypesMetaData, ObsTypes, obs_type,
                   lat, lon, time_arr, lev, observations):
    """
    Collect observations for a single group with efficient numpy operations.
    """
    grp_set = set(obs_group[grp])
    result = {}

    # Build a fast lookup: otyp -> indices in obs_type that match
    for i, otyp in enumerate(ObsTypesMetaData):
        if otyp not in grp_set:
            continue
        
        obs_type_id = ObsTypes[i]
        mask = (obs_type == obs_type_id)
        
        if not mask.any():
            continue
            
        inds = np.where(mask)[0]
        subtyp = otyp.split("_")[0].lower().capitalize()
        
        if subtyp not in result:
            result[subtyp] = {'lat': [], 'lon': [], 'time': [], 'lev': [], 'obs': []}
        
        # Use single indexing operation per group
        result[subtyp]['lat'].append(lat[inds])
        result[subtyp]['lon'].append(lon[inds])
        result[subtyp]['time'].append(time_arr[inds])
        result[subtyp]['lev'].append(lev[inds])
        result[subtyp]['obs'].append(observations[inds])

    return grp, result


if __name__ == "__main__":
    import time as time_module
    start_time = time_module.time()
    
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"

    var_dict = {
        'all': {
            'ObsTypeString': 'all',
            'CopyString': 'observation',
            'QCString': 'DART quality control',
            'nlev': 21,
            'region': '0 360 -90 90 0 120000',
            'verbose': False,
        }
    }

    obs_group = extract_obs_group()
    exp_dict  = extract_exp_info()

    case_name  = 'PNWSNOW'
    resolution = "F20TR_ne30pg2_r05_IcoswISC30E3r5"
    machine    = "compy"
    diag_key   = "obs_seq_final"
    frequency  = "6hourly"
    regnam     = 'global'

    path_template = "/compyfs/zhan391/v3_dart_cda_scratch/%(RUNNAME)s/archive/%(CASENAME)s/dart_diagnostics/%(DIAG)s"
    file_template = "%(RUNNAME)s_%(RES)s_%(MACH)s.dart.e.eam_%(KEY)s.%(TIME)s-*.nc"

    date = "2011-12-03"
    var  = 'all'
    exp  = "DARTEN20"

    # === Resolve paths ===
    run  = exp_dict[exp]['run']
    key  = exp_dict[exp]['key']
    diag = exp_dict[exp].get('diag1', diag_key)
    path = path_template % {'RUNNAME': run, 'CASENAME': key, 'DIAG': diag}
    file = file_template % {'RUNNAME': exp, 'RES': resolution, 'MACH': machine,
                            'KEY': diag_key, 'TIME': date}

    print(f"[{time_module.time() - start_time:.1f}s] Reading observations...")
    (time_arr, lat, lon, lev, which_vert,
     ObsTypesMetaData, ObsTypes, ObsIndex,
     obs_type, obs_keys, qc, observations) = read_dart_obs_seq(
        var, var_dict[var], date, path, file
    )

    print(f"[{time_module.time() - start_time:.1f}s] Filtering unique obs types...")
    unique_obs       = np.unique(obs_type)
    ObsTypesMetaData = ObsTypesMetaData[unique_obs - 1]
    ObsTypes         = ObsTypes[unique_obs - 1]

    print(f"[{time_module.time() - start_time:.1f}s] Collecting obs groups in parallel...")
    delayed_tasks = [
        delayed(_collect_group)(
            grp, obs_group, ObsTypesMetaData, ObsTypes,
            obs_type, lat, lon, time_arr, lev, observations
        )
        for grp in obs_group
    ]

    results = compute(*delayed_tasks)

    data_dict = {}
    for grp, grp_data in results:
        if grp_data:
            data_dict[grp] = grp_data

    print(f"[{time_module.time() - start_time:.1f}s] Plotting observations...")
    draw_obs_location_2d(date, var, var_dict[var], data_dict,
                         fig_path, fgw=20, fgh=12, hs=0.2, ws=0.2)
    
    print(f"[{time_module.time() - start_time:.1f}s] Complete!")

Working on file: /compyfs/zhan391/v3_dart_cda_scratch/DARTEN20_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/dart_en20/dart_diagnostics/obs_seq/DARTEN20_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy.dart.e.eam_obs_seq_final.2011-12-03-*.nc
